# Stage 10.5 (Skid Grouping) — full matrix: Molmo2 vs Qwen (base + adapters) × 3 prompts

One GPU session, everything scored against the same CONSTRUCTED ground truth
(`synth_skid_v1.zip` — real P&ID symbols, dashed skid boundaries drawn deterministically,
the dashed box IS the answer key).

**Reference results already recorded** (same crops, same metric):
- GPT-5.5-low: **91.9%** pairwise (251/273)
- Qwen3-VL base, shared prompt (v1): **44.7%** — merged everything into one group on 5/12
  crops; BELOW the trivial "everyone separate" baseline of 79.1%
- Trivial baselines on this GT: all-separate 79.1%, all-one-group 20.9% — any config below
  79.1% is doing worse than doing nothing

**What this notebook adds, in one Run All:**
1. **Molmo2-O-7B** (better at Stage 4 symbol detection — does its stronger visual grounding
   carry over to reading dashed boundaries?) × 3 prompt variants
2. **Qwen3-VL base** × 3 prompt variants (v1 shared baseline / v2 anti-merge / v3 per-symbol
   assignment reformulation)
3. **Qwen + each existing adapter** (v2 general, v3-stage13, v3-relation) × 3 prompts —
   honest expectation: none was trained on grouping, this is a cheap rule-out, not a bet

Models load SEQUENTIALLY (Molmo2 first, then freed, then Qwen) to fit common Colab GPUs.
transformers==4.57.1 pinned — required by Molmo2's remote-code processor, known-good for
Qwen3-VL (same pin as the Stage 4 detection notebook).

## 1. Config

In [5]:
import os
# Token is read from the environment, never hardcoded. In Colab add it under
# Secrets (key: HF_TOKEN) and enable notebook access; locally just export it.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("HF_TOKEN is not set - add it to Colab Secrets or export it") from e
DATA_REPO = "timthy45/pnid-extraction-datasets"
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MOLMO_MODEL_ID = "allenai/Molmo2-O-7B"

ADAPTERS = {          # name -> path in CKPT_REPO
    "v2-general": "v2/latest",
    "v3-stage13": "v3-stage13/latest",
    "v3-relation": "v3-relation/latest",
}

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "paste-your-hf-token-here"


## 2. Install (transformers pinned for Molmo2's remote-code processor)

In [6]:
!pip install -q transformers==4.57.1 accelerate peft huggingface_hub

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")


CUDA available: True
NVIDIA A100-SXM4-80GB 85 GB


## 3. Download crops + constructed ground truth

In [7]:
import zipfile, time, json, random, re
import multiprocessing as mp
from pathlib import Path
from huggingface_hub import hf_hub_download

def _hf_download_worker(fn_name, kwargs, queue):
    from huggingface_hub import snapshot_download, hf_hub_download
    fn = {"snapshot_download": snapshot_download, "hf_hub_download": hf_hub_download}[fn_name]
    try:
        result = fn(**kwargs)
        queue.put(("ok", result))
    except Exception as e:
        queue.put(("error", str(e)))

def download_with_hard_timeout(fn_name, kwargs, timeout_s=120, max_attempts=8):
    """Unlike load_with_retry/fetch_with_retry, this bounds the wait even when a stalled
    download never raises (silent hang, half-open connection - the Xet-bridge failure mode
    hit repeatedly today). A subprocess can be forcibly terminated; a thread cannot (same
    reason the ModelScope thread-timeout approach was ruled out earlier this session)."""
    for attempt in range(max_attempts):
        q = mp.Queue()
        p = mp.Process(target=_hf_download_worker, args=(fn_name, kwargs, q))
        p.start()
        p.join(timeout=timeout_s)
        if p.is_alive():
            p.terminate()
            p.join()
            print(f"  [hard-timeout retry {attempt+1}/{max_attempts}] stalled >{timeout_s}s - killed, retrying")
            continue
        if q.empty():
            print(f"  [hard-timeout retry {attempt+1}/{max_attempts}] worker died silently, retrying")
            continue
        status, payload = q.get()
        if status == "ok":
            return payload
        print(f"  [hard-timeout retry {attempt+1}/{max_attempts}] {payload}")
    raise RuntimeError(f"download failed after {max_attempts} attempts (hard timeout {timeout_s}s each)")

def fetch_with_retry(filename, timeout_s=120, max_attempts=8):
    # hard-timeout version: a stalled/silently-hung download (no exception, no data) is
    # killed and retried instead of hanging forever.
    return download_with_hard_timeout("hf_hub_download", dict(
        repo_id=DATA_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN),
        timeout_s=timeout_s, max_attempts=max_attempts)

zip_path = fetch_with_retry("benchmarks/synth_skid_v1.zip")
PREP_DIR = Path("/content/synth_skid")
PREP_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(PREP_DIR)

IMG_DIR = PREP_DIR / "images"
with open(PREP_DIR / "synth_meta.json") as f:
    meta = json.load(f)
records = meta["records"]
print(f"loaded {len(records)} crops with constructed ground truth")

import time as _time
import random as _random_retry

def load_with_retry(loader_fn, max_attempts=20, base_backoff_s=10, max_backoff_s=90):
    last_err = None
    for attempt in range(max_attempts):
        try:
            return loader_fn()
        except Exception as e:
            last_err = e
            backoff = min(max_backoff_s, base_backoff_s * (1.5 ** attempt))
            wait = backoff + _random_retry.uniform(0, backoff * 0.3)
            print(f"  [retry {attempt+1}/{max_attempts}] {type(e).__name__}: "
                 f"{str(e)[:160]} - waiting {wait:.0f}s")
            _time.sleep(wait)
    raise RuntimeError(f"failed after {max_attempts} retries") from last_err

from transformers import AutoModelForImageTextToText, AutoProcessor


loaded 12 crops with constructed ground truth


## 4. The 3 prompt variants + scoring harness

- **v1-shared**: the original prompt both GPT-5.5-low (91.9%) and Qwen-base (44.7%) were
  scored on — kept as the controlled baseline.
- **v2-anti-merge**: adds explicit hard negatives against the observed Qwen failure
  (merging everything): "being connected/near does NOT make a group", "default is
  separate", "do not group all symbols together".
- **v3-per-symbol**: reformulates output — instead of emitting groups (where one bad
  decision collapses everything), the model assigns each symbol number to "SKID-1",
  "SKID-2", or "NONE" in a JSON dict. Converted to groups for identical scoring.

In [8]:
PROMPTS = {}

PROMPTS["v1-shared"] = (
    "This P&ID crop has {n} symbols marked with numbered black boxes (0 to {nm1}). Some "
    "symbols are enclosed by a LARGER DASHED colored boundary box labeled SKID-1 or SKID-2 "
    "- this marks a vendor equipment package. Group the symbols: every symbol inside the "
    "same dashed SKID boundary is one group; any symbol NOT inside any dashed boundary is "
    "its own group of one. Respond with ONLY a JSON list of groups, each group a list of "
    "symbol numbers, e.g. [[0,1,2],[3,4],[5]]. Every symbol number 0 to {nm1} must appear "
    "exactly once."
)

PROMPTS["v2-anti-merge"] = (
    "This P&ID crop has {n} symbols marked with numbered black boxes (0 to {nm1}).\n"
    "\n"
    "Some symbols - and ONLY some - are enclosed by a large DASHED rectangle labeled "
    "SKID-1 or SKID-2. Look carefully for a dashed (not solid) rectangle outline; it may "
    "be thin. ONLY symbols physically enclosed inside the same dashed rectangle belong to "
    "the same group.\n"
    "\n"
    "IMPORTANT - do not use any other reason to group symbols:\n"
    "- Being on the same pipe line, being connected by a line, or being near each other "
    "does NOT make two symbols the same group.\n"
    "- Being on the same drawing/sheet does NOT make symbols the same group.\n"
    "- The DEFAULT is that a symbol is its own group of one. Only put symbols together "
    "if they are BOTH inside the SAME dashed rectangle.\n"
    "- Do not group all symbols together. Most crops have 2 small dashed-boundary groups "
    "plus several standalone symbols outside any dashed boundary - not one big group.\n"
    "\n"
    "Respond with ONLY a JSON list of groups, each group a list of symbol numbers, e.g. "
    "[[0,1,2],[3,4],[5]]. Every symbol number 0 to {nm1} must appear exactly once."
)

PROMPTS["v3-per-symbol"] = (
    "This P&ID crop has {n} symbols marked with numbered black boxes (0 to {nm1}). "
    "There are also up to two large DASHED rectangles drawn on the image, labeled SKID-1 "
    "and SKID-2 (dashed outline, not solid). For EACH symbol number, decide: is that "
    "numbered box physically inside the SKID-1 dashed rectangle, inside the SKID-2 dashed "
    "rectangle, or inside neither?\n"
    "\n"
    "Answer with ONLY a JSON object mapping every symbol number to \"SKID-1\", \"SKID-2\", "
    "or \"NONE\", e.g. {{\"0\": \"SKID-1\", \"1\": \"NONE\", \"2\": \"SKID-2\"}}. "
    "Include every number 0 to {nm1} exactly once."
)

def parse_groups_list(text, n):
    m = re.search(r"\[\s*\[.*\]\s*\]", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def parse_per_symbol(text, n):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    buckets = {}
    for k, v in d.items():
        try:
            idx = int(k)
        except (ValueError, TypeError):
            continue
        buckets.setdefault(str(v).strip().upper(), []).append(idx)
    groups = []
    for label, members in buckets.items():
        if label in ("SKID-1", "SKID-2"):
            groups.append(sorted(members))
        else:
            groups.extend([[i] for i in members])
    return groups

def parse_any(text, n, prompt_name):
    parsed = parse_per_symbol(text, n) if prompt_name == "v3-per-symbol" else parse_groups_list(text, n)
    if parsed is None:
        return [[i] for i in range(n)], True   # fallback + parse-failure flag
    return parsed, False

def groups_to_pair_labels(groups, n):
    group_of = {}
    for gi, g in enumerate(groups):
        for i in g:
            if 0 <= int(i) < n:
                group_of[int(i)] = gi
    labels = {}
    for i in range(n):
        for j in range(i + 1, n):
            labels[(i, j)] = (i in group_of and group_of.get(i) == group_of.get(j))
    return labels

def score_config(generate_fn, config_name):
    per_prompt = {}
    for pname, ptmpl in PROMPTS.items():
        total_pairs = correct_pairs = parse_fails = 0
        preds = {}
        for rec in records:
            sheet_id, n = rec["sheet_id"], rec["n"]
            from PIL import Image
            img = Image.open(IMG_DIR / f"{sheet_id}.png")
            prompt = ptmpl.format(n=n, nm1=n - 1)
            raw = generate_fn(img, prompt)
            pred_groups, failed = parse_any(raw, n, pname)
            parse_fails += failed
            preds[sheet_id] = pred_groups
            gt_labels = groups_to_pair_labels(rec["gt_groups"], n)
            pred_labels = groups_to_pair_labels(pred_groups, n)
            for pair, gt_same in gt_labels.items():
                total_pairs += 1
                if pred_labels.get(pair, False) == gt_same:
                    correct_pairs += 1
        acc = correct_pairs / total_pairs if total_pairs else 0
        per_prompt[pname] = {"pairwise_acc": acc, "correct": correct_pairs,
                             "total": total_pairs, "parse_failures": parse_fails,
                             "predictions": preds}
        print(f"  {config_name} × {pname}: {correct_pairs}/{total_pairs} = {acc:.1%}"
             f"{'  (parse fails: ' + str(parse_fails) + ')' if parse_fails else ''}")
    return per_prompt

all_results = {}
print("harness ready - reference: GPT-5.5-low 91.9%, all-separate baseline 79.1%")


harness ready - reference: GPT-5.5-low 91.9%, all-separate baseline 79.1%


## 5. Molmo2-O-7B — load, score all 3 prompts, free VRAM

In [5]:
from transformers import AutoModelForImageTextToText, AutoProcessor

molmo_processor = load_with_retry(lambda: AutoProcessor.from_pretrained(
    MOLMO_MODEL_ID, trust_remote_code=True, dtype="auto"))
molmo_model = load_with_retry(lambda: AutoModelForImageTextToText.from_pretrained(
    MOLMO_MODEL_ID, trust_remote_code=True, dtype="auto", device_map="cuda"))
print("Molmo2 loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

def molmo_generate(image, prompt, max_tokens=600):
    messages = [{"role": "user", "content": [
        {"type": "text", "text": prompt}, {"type": "image", "image": image}]}]
    inputs = molmo_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(molmo_model.device)
    with torch.no_grad():
        out = molmo_model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[1]:]
    return molmo_processor.batch_decode(gen, skip_special_tokens=True)[0].strip()

print("\n=== Molmo2-O-7B ===")
all_results["molmo2"] = score_config(molmo_generate, "molmo2")

del molmo_model, molmo_processor
torch.cuda.empty_cache()
print("Molmo2 freed. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

processing_molmo2.py: 0.00B [00:00, ?B/s]

video_processing_molmo2.py: 0.00B [00:00, ?B/s]

...

model-00001-of-00007.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00007-of-00007.safetensors:   0%|          | 0.00/1.87G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Molmo2 loaded. VRAM: 31.0 GB

=== Molmo2-O-7B ===
  molmo2 × v1-shared: 57/273 = 20.9%
  molmo2 × v2-anti-merge: 57/273 = 20.9%
  molmo2 × v3-per-symbol: 216/273 = 79.1%
Molmo2 freed. VRAM: 0.0 GB


## 6. Qwen3-VL base + the 3 existing adapters — score all, one load

In [9]:
from peft import PeftModel
from huggingface_hub import snapshot_download

qwen_processor = load_with_retry(lambda: AutoProcessor.from_pretrained(QWEN_MODEL_ID))
qwen_base = load_with_retry(lambda: AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")).eval()
print("Qwen base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

def _pull(adapter_path):
    # hard-timeout version - see download_with_hard_timeout in the data-download cell above.
    # load_with_retry alone never catches a silent stall (no exception raised), which is
    # exactly the failure mode this hit live during benchmarking.
    local = Path(f"/content/adp_{adapter_path.replace('/', '_')}")
    download_with_hard_timeout("snapshot_download", dict(
        repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
        allow_patterns=[f"{adapter_path}/*"], local_dir=str(local)))
    d = local / adapter_path
    assert (d / "adapter_model.safetensors").exists(), f"missing: {d}"
    return str(d)

adapter_names = list(ADAPTERS)
first = adapter_names[0]
qwen_model = PeftModel.from_pretrained(qwen_base, _pull(ADAPTERS[first]), adapter_name=first)
for name in adapter_names[1:]:
    qwen_model.load_adapter(_pull(ADAPTERS[name]), adapter_name=name)
qwen_model.eval()
print(f"adapters attached: {adapter_names}")

def qwen_generate(image, prompt, max_tokens=600):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = qwen_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(qwen_model.device)
    with torch.no_grad():
        out = qwen_model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    t = out[0][inputs["input_ids"].shape[1]:]
    return qwen_processor.decode(t, skip_special_tokens=True).strip()

# base first (adapters disabled), then each adapter
print("\n=== Qwen3-VL base (no adapter) ===")
with qwen_model.disable_adapter():
    all_results["qwen-base"] = score_config(qwen_generate, "qwen-base")

for name in adapter_names:
    print(f"\n=== Qwen + {name} ===")
    qwen_model.set_adapter(name)
    all_results[f"qwen+{name}"] = score_config(qwen_generate, f"qwen+{name}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen base loaded. VRAM: 17.5 GB
  [hard-timeout retry 1/8] stalled >120s - killed, retrying
  [hard-timeout retry 2/8] stalled >120s - killed, retrying
  [hard-timeout retry 3/8] stalled >120s - killed, retrying
  [hard-timeout retry 4/8] stalled >120s - killed, retrying
  [hard-timeout retry 5/8] stalled >120s - killed, retrying
  [hard-timeout retry 6/8] stalled >120s - killed, retrying
  [hard-timeout retry 7/8] stalled >120s - killed, retrying
  [hard-timeout retry 8/8] stalled >120s - killed, retrying


RuntimeError: download failed after 8 attempts (hard timeout 120s each)

## 7. Summary table + push to HF, then disconnect

In [ ]:
print(f"{'config':26s}" + "".join(f"{p:>16s}" for p in PROMPTS))
print("-" * (26 + 16 * len(PROMPTS)))
for config, per_prompt in all_results.items():
    row = f"{config:26s}"
    for pname in PROMPTS:
        acc = per_prompt[pname]["pairwise_acc"]
        row += f"{acc:>15.1%} "
    print(row)
print("-" * (26 + 16 * len(PROMPTS)))
print(f"{'REFERENCE gpt-5.5-low':26s}{'91.9%':>15s}  (v1-shared prompt, scored locally)")
print(f"{'BASELINE all-separate':26s}{'79.1%':>15s}  (trivial - any config below this is net-negative)")

slim = {config: {pname: {k: v for k, v in pdata.items() if k != "predictions"}
                for pname, pdata in per_prompt.items()}
        for config, per_prompt in all_results.items()}
full = {"summary": slim, "predictions": {config: {pname: pdata["predictions"]
        for pname, pdata in per_prompt.items()} for config, per_prompt in all_results.items()}}

with open("/content/skid_matrix_results.json", "w") as f:
    json.dump(full, f, indent=2)

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="/content/skid_matrix_results.json",
    path_in_repo="benchmarks/skid_matrix_molmo2_qwen_adapters_v1.json",
    repo_id=DATA_REPO, repo_type="dataset", token=HF_TOKEN)
print("\nresults pushed to HF")

from google.colab import runtime
runtime.unassign()


In [10]:
import time, requests

print("=== 1. baseline network speed (unrelated CDN) ===")
t0 = time.time()
r = requests.get("https://speed.cloudflare.com/__down?bytes=25000000", timeout=30)
dt = time.time() - t0
print(f"{len(r.content)/1e6:.1f} MB in {dt:.1f}s = {len(r.content)/1e6/dt:.2f} MB/s")

print("\n=== 2. resolve the actual adapter file URL + redirect target ===")
from huggingface_hub import hf_hub_url, get_hf_file_metadata
url = hf_hub_url("timthy45/qwen3vl-pnid-domain-base", "v3-stage13/latest/adapter_model.safetensors", repo_type="model")
print("resolve URL:", url)
meta = get_hf_file_metadata(url, token=HF_TOKEN)
print("redirect target:", meta.location[:120] if meta.location else None)
print("expected size:", meta.size)

print("\n=== 3. direct timed GET against that redirect target, first 20MB only ===")
t0 = time.time()
resp = requests.get(meta.location, headers={"Range": "bytes=0-20000000"}, timeout=60, stream=True)
got = 0
for chunk in resp.iter_content(chunk_size=1024*1024):
    got += len(chunk)
    if time.time() - t0 > 45:
        print(f"  still going after 45s, got {got/1e6:.1f} MB so far")
        break
dt = time.time() - t0
print(f"got {got/1e6:.1f} MB in {dt:.1f}s = {got/1e6/dt:.2f} MB/s, http_status={resp.status_code}")

=== 1. baseline network speed (unrelated CDN) ===
25.0 MB in 0.2s = 151.27 MB/s

=== 2. resolve the actual adapter file URL + redirect target ===
resolve URL: https://huggingface.co/timthy45/qwen3vl-pnid-domain-base/resolve/main/v3-stage13/latest/adapter_model.safetensors
redirect target: https://cas-bridge.xethub.hf.co/xet-bridge-us/6a55b238d13da0ab2d931765/b18d649f5bb0f95e3d612c57ad5c3a1b69763f5c9bb0091e2
expected size: 1429309304

=== 3. direct timed GET against that redirect target, first 20MB only ===
got 20.0 MB in 0.1s = 186.35 MB/s, http_status=206


In [13]:
import json

print(f"{'config':26s}" + "".join(f"{p:>16s}" for p in PROMPTS))
print("-" * (26 + 16 * len(PROMPTS)))
for config, per_prompt in all_results.items():
    row = f"{config:26s}"
    for pname in PROMPTS:
        acc = per_prompt[pname]["pairwise_acc"]
        row += f"{acc:>15.1%} "
    print(row)
print("-" * (26 + 16 * len(PROMPTS)))
print(f"{'REFERENCE gpt-5.5-low':26s}{'91.9%':>15s}  (v1-shared prompt, scored locally)")
print(f"{'BASELINE all-separate':26s}{'79.1%':>15s}  (trivial - any config below this is net-negative)")

slim = {config: {pname: {k: v for k, v in pdata.items() if k != "predictions"}
                for pname, pdata in per_prompt.items()}
        for config, per_prompt in all_results.items()}
full = {"summary": slim, "predictions": {config: {pname: pdata["predictions"]
        for pname, pdata in per_prompt.items()} for config, per_prompt in all_results.items()}}

with open("/content/skid_matrix_results.json", "w") as f:
    json.dump(full, f, indent=2)

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="/content/skid_matrix_results.json",
    path_in_repo="benchmarks/skid_matrix_molmo2_qwen_adapters_v1.json",
    repo_id=DATA_REPO, repo_type="dataset", token=HF_TOKEN)
print("\nresults pushed to HF")

from google.colab import runtime
runtime.unassign()

config                           v1-shared   v2-anti-merge   v3-per-symbol
--------------------------------------------------------------------------
qwen-base                           47.6%           56.4%           92.3% 
qwen+v2-general                     79.1%           79.1%           79.1% 
qwen+v3-stage13                     26.0%           24.5%           61.5% 
qwen+v3-relation                    79.1%           79.1%           79.1% 
--------------------------------------------------------------------------
REFERENCE gpt-5.5-low               91.9%  (v1-shared prompt, scored locally)
BASELINE all-separate               79.1%  (trivial - any config below this is net-negative)

results pushed to HF


In [11]:
import requests
from pathlib import Path
from huggingface_hub import hf_hub_url

def _pull(adapter_path):
    local = Path(f"/content/adp_{adapter_path.replace('/', '_')}") / adapter_path
    local.mkdir(parents=True, exist_ok=True)
    for fname in ["adapter_config.json", "adapter_model.safetensors"]:
        dest = local / fname
        if dest.exists() and dest.stat().st_size > 0:
            continue
        url = hf_hub_url("timthy45/qwen3vl-pnid-domain-base", f"{adapter_path}/{fname}", repo_type="model")
        r = requests.get(url, headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=60, stream=True, allow_redirects=True)
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                f.write(chunk)
        print(f"  fetched {fname} ({dest.stat().st_size/1e6:.1f} MB)")
    assert (local / "adapter_model.safetensors").exists(), f"missing: {local}"
    return str(local)

In [12]:
from peft import PeftModel

adapter_names = list(ADAPTERS)
first = adapter_names[0]
qwen_model = PeftModel.from_pretrained(qwen_base, _pull(ADAPTERS[first]), adapter_name=first)
for name in adapter_names[1:]:
    qwen_model.load_adapter(_pull(ADAPTERS[name]), adapter_name=name)
qwen_model.eval()
print(f"adapters attached: {adapter_names}")

def qwen_generate(image, prompt, max_tokens=600):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = qwen_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(qwen_model.device)
    with torch.no_grad():
        out = qwen_model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    t = out[0][inputs["input_ids"].shape[1]:]
    return qwen_processor.decode(t, skip_special_tokens=True).strip()

print("\n=== Qwen3-VL base (no adapter) ===")
with qwen_model.disable_adapter():
    all_results["qwen-base"] = score_config(qwen_generate, "qwen-base")

for name in adapter_names:
    print(f"\n=== Qwen + {name} ===")
    qwen_model.set_adapter(name)
    all_results[f"qwen+{name}"] = score_config(qwen_generate, f"qwen+{name}")

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


  fetched adapter_model.safetensors (1429.3 MB)


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


  fetched adapter_config.json (0.0 MB)
  fetched adapter_model.safetensors (1429.3 MB)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


adapters attached: ['v2-general', 'v3-stage13', 'v3-relation']

=== Qwen3-VL base (no adapter) ===
  qwen-base × v1-shared: 130/273 = 47.6%
  qwen-base × v2-anti-merge: 154/273 = 56.4%
  qwen-base × v3-per-symbol: 252/273 = 92.3%

=== Qwen + v2-general ===
  qwen+v2-general × v1-shared: 216/273 = 79.1%  (parse fails: 12)
  qwen+v2-general × v2-anti-merge: 216/273 = 79.1%  (parse fails: 12)
  qwen+v2-general × v3-per-symbol: 216/273 = 79.1%  (parse fails: 12)

=== Qwen + v3-stage13 ===
  qwen+v3-stage13 × v1-shared: 71/273 = 26.0%
  qwen+v3-stage13 × v2-anti-merge: 67/273 = 24.5%
  qwen+v3-stage13 × v3-per-symbol: 168/273 = 61.5%

=== Qwen + v3-relation ===
  qwen+v3-relation × v1-shared: 216/273 = 79.1%  (parse fails: 12)
  qwen+v3-relation × v2-anti-merge: 216/273 = 79.1%  (parse fails: 12)
  qwen+v3-relation × v3-per-symbol: 216/273 = 79.1%  (parse fails: 12)
